# Exercise 7: From the log to the lake

Every run of your job does three things: **read** the new records of the topic `logs`,
**write** them into the lake (Parquet files in `data/lake/`), **remember** where it stopped.

This notebook gives you the building blocks (`tools/lakelib.py`). You write one function,
`run_once()`. Save your final version of this notebook into your `solution/` folder.

In [2]:
import sys; sys.path.append("/home/jovyan/tools")
from lakelib import *

(installing confluent-kafka==2.11.0 ...)
(installing duckdb==1.5.6 ...)


## 1. The example: a job that copies everything

It reads the topic from the beginning, every time. Run the next two cells and look at the
checker's last line.

In [3]:
def run_once():
    new = read_new(None)          # None = from the beginning of the topic
    write_files(new, LAKE)        # straight into the lake

reset()                           # empty lake
run_once()
check()

reset: lake, staging and state deleted
read_new: 2,000,000 new records
write_files: 2,000,000 records -> /home/jovyan/data/lake/logs/run-20261119-105237-c764bd

Lake check — topic 'logs' (4 partitions)   2026-11-19 10:54:11
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 502,741)     502,741           0          0           0
        1           [0, 493,273)     493,273           0          0           0
        2           [0, 478,852)     478,852           0          0           0
        3           [0, 525,134)     525,134           0          0           0
-------------------------------------------------------------------------------
    total      2,000,000 records   2,000,000           0          0           0

files under /home/jovyan/data/lake: 4 data files, 111.4 MB, 27.85 MB pe

{'topic': 2000000,
 'lake': 2000000,
 'dup': 0,
 'miss': 0,
 'lag': 0,
 'files': 4,
 'mb': 111.395469,
 'ok': True,
 'problems': []}

In [4]:
run_once()                        # the next run
check()

read_new: 2,000,000 new records
write_files: 2,000,000 records -> /home/jovyan/data/lake/logs/run-20261119-140719-3371b8

Lake check — topic 'logs' (4 partitions)   2026-11-19 14:08:23
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 502,741)   1,005,482     502,741          0           0
        1           [0, 493,273)     986,546     493,273          0           0
        2           [0, 478,852)     957,704     478,852          0           0
        3           [0, 525,134)   1,050,268     525,134          0           0
-------------------------------------------------------------------------------
    total      2,000,000 records   4,000,000   2,000,000          0           0

files under /home/jovyan/data/lake: 8 data files, 222.8 MB, 27.85 MB per file on average
check took 0.9 s

Exa

{'topic': 2000000,
 'lake': 4000000,
 'dup': 2000000,
 'miss': 0,
 'lag': 0,
 'files': 8,
 'mb': 222.790938,
 'ok': False,
 'problems': ['2,000,000 duplicates']}

## 2. The building blocks

| Block | What it does |
|---|---|
| `load_state()` | where the last run stopped, from a small state file (`None` if there is none) |
| `save_state(new.end)` | stores where this run stopped in the state file |
| `lake_position()` | where the last run stopped, read from the lake itself (`None` if the lake is empty) |
| `read_new(start)` | the records after `start`, up to the end of the topic *now* (`new.end`) |
| `write_files(new, folder)` | writes them as Parquet into a new run folder inside `LAKE` or `STAGING` |
| `publish(run)` | moves a finished run folder from `STAGING` into the lake: one rename, a single step |

`LAKE` is what readers see; `STAGING` is a folder next to it that nobody reads.
`new.df.show()` shows the records of a run as a Spark DataFrame.

## 3. Your architecture

Write `run_once()` for the architecture you chose (A, B or C, see the README).
The order of the calls *is* your architecture.

In [ ]:
def run_once():
    # 1. where did the last run stop?
    start = lake_position()
    # 2. read what is new
    new = read_new(start)
    # 3. write it
    run = write_files(new, STAGING)
    # 4. publish: one rename moves it into the lake
    publish(run)

## 4. Check it

`check_all` empties the lake, then runs your job in five situations: a first run, new records,
nothing new, a run that stops **while** writing, a run that stops **after** writing.
It takes 2-4 minutes.

Architecture C: readers use a view, so check through it: `check_all(run_once, lake_sql=DEDUP_SQL)`.

In [16]:
check_all(run_once)

reset: lake, staging and state deleted

situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             2,600,000           0        0           0 103.8s  PASS
2 new records arrive, run again         2,700,000           0        0           0  15.9s  PASS
3 nothing new, run again                2,700,000           0        0           0   1.0s  PASS
4 run stops WHILE writing: right after  2,700,000           0        0     100,000         PASS
  ... then the next run                 2,800,000           0        0           0   9.4s  PASS
5 run stops AFTER writing: right after  2,800,000           0        0     100,000         PASS
  ... then the next run                 2,900,000           0        0           0  11.9s  PASS
-----------------------------------------------------------------------------------------------

[('1 first run',
  {'topic': 2600000,
   'lake': 2600000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 121.739291,
   'ok': True,
   'problems': []},
  103.77388954162598),
 ('2 new records arrive, run again',
  {'topic': 2700000,
   'lake': 2700000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 122.898396,
   'ok': True,
   'problems': []},
  15.925145626068115),
 ('3 nothing new, run again',
  {'topic': 2700000,
   'lake': 2700000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 122.898396,
   'ok': True,
   'problems': []},
  0.991300106048584),
 ('4 run stops WHILE writing: right after',
  {'topic': 2800000,
   'lake': 2700000,
   'dup': 0,
   'miss': 0,
   'lag': 100000,
   'files': 8,
   'mb': 122.898396,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 2800000,
   'lake': 2800000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 12,
   'mb': 124.058328,
   'ok': True,
   'problems': []},
